# Project Odin — NNUE Training (Kaggle GPU)

Self-contained training notebook for the OdinNNUE 4-player chess evaluation network.

## Setup
1. Upload your `.bin` training data file as a Kaggle Dataset (or use "Add Data")
2. Enable **GPU** accelerator (Settings > Accelerator > GPU T4 x2 or P100)
3. Set `BIN_PATH` below to your uploaded data file path
4. Run all cells
5. Download the output `.pt` model file when done

## Architecture
- **Input**: HalfKP-4 features (4,480 per perspective, 4 perspectives)
- **Feature Transformer**: 4,480 → 256 (SCReLU) ×4 perspectives
- **Hidden**: 1,024 → 32 (ReLU)
- **Dual Heads**: BRS (32→1 centipawns) + MCTS (32→4 per-player values)
- **Loss**: Multi-task (BRS MSE + MCTS blended + Game Result)

In [ ]:
# === CONFIGURATION ===
# Update BIN_PATH to point to your uploaded .bin training data.
# On Kaggle, uploaded datasets are at /kaggle/input/<dataset-name>/
BIN_PATH = '/kaggle/input/odin-gen0-training-data/training_data_gen0.bin'
MODEL_OUTPUT = '/kaggle/working/best_model.pt'
ONNUE_OUTPUT = '/kaggle/working/weights_gen0.onnue'

In [ ]:
import struct
import binascii
import sys
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Definition

In [ ]:
class SCReLU(nn.Module):
    """Squared Clipped ReLU: clamp(x, 0, QA)^2"""
    def __init__(self, qa=255.0):
        super().__init__()
        self.qa = qa

    def forward(self, x):
        return torch.clamp(x, 0.0, self.qa) ** 2


class OdinNNUE(nn.Module):
    def __init__(self, num_features=4480, ft_out=256, hidden=32):
        super().__init__()
        self.num_features = num_features
        self.ft_out = ft_out
        self.qa = 255.0

        # 4 separate feature transformers (one per perspective)
        self.ft = nn.ModuleList([
            nn.Linear(num_features, ft_out) for _ in range(4)
        ])
        self.screlu = SCReLU(self.qa)

        # Hidden layer: 4*256 = 1024 -> 32
        self.hidden = nn.Linear(4 * ft_out, hidden)

        # Dual output heads
        self.brs_head = nn.Linear(hidden, 1)
        self.mcts_head = nn.Linear(hidden, 4)

    def forward(self, features):
        ft_outs = []
        for p in range(4):
            ft_outs.append(self.screlu(self.ft[p](features[:, p, :])))
        concat = torch.cat(ft_outs, dim=1)
        concat = concat / self.qa
        h = torch.relu(self.hidden(concat))
        brs_out = self.brs_head(h)
        mcts_out = self.mcts_head(h)
        return brs_out, mcts_out

print(f'OdinNNUE parameters: {sum(p.numel() for p in OdinNNUE().parameters()):,}')

## Dataset Loader

In [ ]:
SAMPLE_SIZE = 556
PERSPECTIVE_BYTES = 129  # 1 byte count + 64 * 2 byte indices

class OdinDataset(Dataset):
    def __init__(self, bin_path, num_features=4480):
        with open(bin_path, 'rb') as f:
            self.data = f.read()
        self.num_samples = len(self.data) // SAMPLE_SIZE
        self.num_features = num_features

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        offset = idx * SAMPLE_SIZE
        features = torch.zeros(4, self.num_features)
        for p in range(4):
            base = offset + p * PERSPECTIVE_BYTES
            count = self.data[base]
            for i in range(count):
                feat_idx = struct.unpack_from('<H', self.data, base + 1 + i * 2)[0]
                if feat_idx < self.num_features:
                    features[p, feat_idx] = 1.0
        target_offset = offset + 516
        brs_target = struct.unpack_from('<h', self.data, target_offset)[0]
        mcts_targets = struct.unpack_from('<4f', self.data, target_offset + 2)
        game_result = struct.unpack_from('<4f', self.data, target_offset + 18)
        return (
            features,
            torch.tensor(brs_target, dtype=torch.float32),
            torch.tensor(mcts_targets, dtype=torch.float32),
            torch.tensor(game_result, dtype=torch.float32),
        )

dataset = OdinDataset(BIN_PATH)
print(f'Loaded {len(dataset):,} samples from {BIN_PATH}')
print(f'File size: {len(dataset.data) / 1024 / 1024:.1f} MB')

## Training Loop

In [ ]:
# Hyperparameters
BATCH_SIZE = 4096
LR = 0.01
EPOCHS = 20
LAMBDA_BRS = 1.0
LAMBDA_MCTS = 0.5
LAMBDA_RESULT = 0.25
OUTPUT_SCALE = 400.0
SIGMOID_K = 4000.0


def compute_loss(model, features, brs_target, mcts_target, game_result):
    brs_pred, mcts_pred = model(features)
    brs_loss = nn.functional.mse_loss(
        brs_pred.squeeze() / OUTPUT_SCALE,
        brs_target / OUTPUT_SCALE
    )
    mcts_blended = 0.7 * mcts_target + 0.3 * game_result
    mcts_pred_sigmoid = torch.sigmoid(mcts_pred / SIGMOID_K)
    mcts_loss = nn.functional.mse_loss(mcts_pred_sigmoid, mcts_blended)
    result_pred = torch.sigmoid(mcts_pred / SIGMOID_K)
    result_loss = nn.functional.mse_loss(result_pred, game_result)
    total = LAMBDA_BRS * brs_loss + LAMBDA_MCTS * mcts_loss + LAMBDA_RESULT * result_loss
    return total, brs_loss, mcts_loss, result_loss


# Data split
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {train_size:,} samples ({len(train_loader)} batches)')
print(f'Val:   {val_size:,} samples ({len(val_loader)} batches)')

In [ ]:
model = OdinNNUE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_val_loss = float('inf')
patience = 5
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for features, brs_target, mcts_target, game_result in train_loader:
        features = features.to(device)
        brs_target = brs_target.to(device)
        mcts_target = mcts_target.to(device)
        game_result = game_result.to(device)

        loss, _, _, _ = compute_loss(model, features, brs_target, mcts_target, game_result)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for features, brs_target, mcts_target, game_result in val_loader:
            features = features.to(device)
            brs_target = brs_target.to(device)
            mcts_target = mcts_target.to(device)
            game_result = game_result.to(device)
            loss, _, _, _ = compute_loss(model, features, brs_target, mcts_target, game_result)
            val_loss += loss.item()

    avg_train = total_loss / max(len(train_loader), 1)
    avg_val = val_loss / max(len(val_loader), 1)
    lr = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch+1}/{EPOCHS}  train_loss={avg_train:.6f}  val_loss={avg_val:.6f}  lr={lr:.6f}')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        patience_counter = 0
        # Save to CPU so export works without GPU
        cpu_state = {k: v.cpu() for k, v in model.state_dict().items()}
        torch.save(cpu_state, MODEL_OUTPUT)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

    scheduler.step()

print(f'\nBest validation loss: {best_val_loss:.6f}')
print(f'Model saved to {MODEL_OUTPUT}')

## Export to .onnue

Quantizes the trained model to the binary `.onnue` format that the Rust engine loads.
Download both `best_model.pt` and `weights_gen0.onnue` from the output.

In [ ]:
# Constants — must match Rust exactly
ONNUE_MAGIC = b'ONUE'
ONNUE_VERSION = 1
QA = 255
QB = 64
FEATURES = 4480
FT_OUT = 256
HIDDEN = 32


def architecture_hash():
    descriptor = f'HalfKP4-{FEATURES}-{FT_OUT}-{HIDDEN}-1-4'
    desc_bytes = descriptor.encode('ascii')
    result = bytearray(32)
    for chunk_idx in range(4):
        h = (0xcbf29ce484222325 + chunk_idx) & 0xFFFFFFFFFFFFFFFF
        for b in desc_bytes:
            h ^= b
            h = (h * 0x00000100000001b3) & 0xFFFFFFFFFFFFFFFF
        result[chunk_idx * 8:chunk_idx * 8 + 8] = h.to_bytes(8, 'little')
    return bytes(result)


def crc32_ieee(data):
    return binascii.crc32(data) & 0xFFFFFFFF


def quantize_ft(weight, bias, qa=QA):
    w_max = max(weight.abs().max().item(), 1e-6)
    scale = min(qa / w_max, 32767.0 / w_max)
    w_q = torch.clamp(torch.round(weight * scale), -32768, 32767).to(torch.int16)
    b_q = torch.clamp(torch.round(bias * scale), -32768, 32767).to(torch.int16)
    return w_q, b_q


def quantize_hidden(weight, bias):
    w_max = max(weight.abs().max().item(), 1e-6)
    scale = min(127.0 / w_max, 127.0)
    w_q = torch.clamp(torch.round(weight * scale), -128, 127).to(torch.int8)
    b_q = torch.round(bias * scale).to(torch.int32)
    return w_q, b_q


# Load trained model and export
export_model = OdinNNUE()
export_model.load_state_dict(torch.load(MODEL_OUTPUT, map_location='cpu', weights_only=True))
export_model.eval()

buf = bytearray()
buf += ONNUE_MAGIC
buf += struct.pack('<I', ONNUE_VERSION)
buf += architecture_hash()
buf += struct.pack('<I', FEATURES)
buf += struct.pack('<I', FT_OUT)

for p in range(4):
    w = export_model.ft[p].weight.detach()
    b = export_model.ft[p].bias.detach()
    w_q, b_q = quantize_ft(w, b)
    for feat in range(FEATURES):
        for neuron in range(FT_OUT):
            buf += struct.pack('<h', w_q[neuron, feat].item())
    for neuron in range(FT_OUT):
        buf += struct.pack('<h', b_q[neuron].item())

w = export_model.hidden.weight.detach()
b = export_model.hidden.bias.detach()
w_q, b_q = quantize_hidden(w, b)
for inp in range(4 * FT_OUT):
    for neuron in range(HIDDEN):
        buf += struct.pack('<b', w_q[neuron, inp].item())
for neuron in range(HIDDEN):
    buf += struct.pack('<i', b_q[neuron].item())

w = export_model.brs_head.weight.detach()
b = export_model.brs_head.bias.detach()
w_q, b_q = quantize_hidden(w, b)
for h in range(HIDDEN):
    buf += struct.pack('<b', w_q[0, h].item())
buf += struct.pack('<i', b_q[0].item())

w = export_model.mcts_head.weight.detach()
b = export_model.mcts_head.bias.detach()
w_q, b_q = quantize_hidden(w, b)
for h in range(HIDDEN):
    for v in range(4):
        buf += struct.pack('<b', w_q[v, h].item())
for v in range(4):
    buf += struct.pack('<i', b_q[v].item())

checksum = crc32_ieee(bytes(buf))
buf += struct.pack('<I', checksum)

with open(ONNUE_OUTPUT, 'wb') as f:
    f.write(buf)

print(f'Exported {ONNUE_OUTPUT} ({len(buf):,} bytes)')
print(f'CRC32: {checksum:#010x}')
print(f'\nDownload both files from /kaggle/working/:')
print(f'  - {MODEL_OUTPUT} (PyTorch state dict)')
print(f'  - {ONNUE_OUTPUT} (quantized weights for Rust engine)')